# AgentCore Gateway + External Neo4j MCP Test

## Prerequisites

- Deploy the stack first: `cdk deploy Neo4jAgentCoreGatewayStack -c domain_name=... -c certificate_arn=...`
- Ensure you are authenticated with AWS (e.g. `aws sso login --profile YOUR_PROFILE`)
- Fill in `STACK_NAME` below, or set the gateway variables manually

In [1]:
STACK_NAME = "Neo4jAgentCoreGatewayStack"

In [2]:
import boto3
import json
import requests

cfn = boto3.client("cloudformation")
outputs = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}

GATEWAY_URL = outputs["GatewayUrl"]
COGNITO_TOKEN_ENDPOINT = outputs["CognitoTokenEndpoint"]
COGNITO_CLIENT_ID = outputs["CognitoAppClientId"]
COGNITO_SCOPE = outputs["CognitoScope"]
COGNITO_USER_POOL_ID = outputs["CognitoUserPoolId"]

cognito = boto3.client("cognito-idp")
client_secret = cognito.describe_user_pool_client(
    UserPoolId=COGNITO_USER_POOL_ID,
    ClientId=COGNITO_CLIENT_ID,
)["UserPoolClient"]["ClientSecret"]

token_resp = requests.post(
    COGNITO_TOKEN_ENDPOINT,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": COGNITO_CLIENT_ID,
        "client_secret": client_secret,
        "scope": COGNITO_SCOPE,
    },
)
token_resp.raise_for_status()
ACCESS_TOKEN = token_resp.json()["access_token"]
print("OAuth token obtained.")

OAuth token obtained.


In [3]:
def gateway_request(method: str, params: dict | None = None):
    payload = {"jsonrpc": "2.0", "id": "demo", "method": method}
    if params is not None:
        payload["params"] = params
    r = requests.post(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Content-Type": "application/json",
        },
        json=payload,
    )
    r.raise_for_status()
    return r.json()

In [4]:
result = gateway_request("tools/list")
tools = result.get("result", {}).get("tools", result.get("result", []))
if isinstance(tools, list):
    for t in tools:
        name = t.get("name", t) if isinstance(t, dict) else str(t)
        desc = t.get("description", "") if isinstance(t, dict) else ""
        print(f"- {name}: {desc}")
else:
    print(json.dumps(result, indent=2))

- neo4j-mcp___get-schema: 
		Retrieve the schema information from the Neo4j database, including node labels, relationship types, and property keys.
		If the database contains no data, no schema information is returned.
- neo4j-mcp___read-cypher: read-cypher can run only read-only Cypher statements. For write operations (CREATE, MERGE, DELETE, SET, etc...), schema/admin commands, or PROFILE queries, use write-cypher instead.


Call a Neo4j MCP tool (e.g. `read_schema` or `run_cypher`). Adjust tool name and arguments to match your deployed Neo4j MCP.

In [6]:
tool_result = gateway_request("tools/call", {
    "name": "neo4j-mcp___get-schema",
    "arguments": {},
})
print(json.dumps(tool_result, indent=2))

{
  "jsonrpc": "2.0",
  "id": "demo",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "[{\"key\":\"_Bloom_Perspective_\",\"value\":{\"type\":\"node\",\"properties\":{\"data\":\"STRING\",\"id\":\"STRING\",\"name\":\"STRING\",\"roles\":\"LIST\",\"version\":\"STRING\"},\"relationships\":{\"_Bloom_HAS_SCENE_\":{\"direction\":\"out\",\"labels\":[\"_Bloom_Scene_\"]}}}},{\"key\":\"IndustryCategory\",\"value\":{\"type\":\"node\",\"properties\":{\"id\":\"STRING\",\"name\":\"STRING\"},\"relationships\":{\"HAS_CATEGORY\":{\"direction\":\"in\",\"labels\":[\"Organization\"]}}}},{\"key\":\"HAS_CEO\",\"value\":{\"type\":\"relationship\"}},{\"key\":\"IN_COUNTRY\",\"value\":{\"type\":\"relationship\"}},{\"key\":\"Fewshot\",\"value\":{\"type\":\"node\",\"properties\":{\"Cypher\":\"STRING\",\"Question\":\"STRING\",\"embedding\":\"LIST\",\"id\":\"INTEGER\"}}},{\"key\":\"Organization\",\"value\":{\"type\":\"node\",\"properties\":{\"diffbotId\":\"STRING\",\"id\":\"STRING\",\"i

In [9]:
tool_result = gateway_request("tools/call", {
    "name": "neo4j-mcp___read-cypher",
    "arguments": {"query": "RETURN 1 AS n"},
})
print(json.dumps(tool_result, indent=2))

{
  "jsonrpc": "2.0",
  "id": "demo",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "[\n  {\n    \"n\": 1\n  }\n]"
      }
    ]
  }
}
